In [19]:
import os
from pathlib import Path

import pandas as pd
import numpy as np

from src.preprocess.model import StorageStaticParams, InputData



### Capacity Aging Linear Approximation

The capacity in energy storage decrease over time. It decreases with aging, but also decreases with each performed cycle. What I want to do is to approximate this phenomena by a model, that can be injected into MILP optimization formulation.

Goal is to find formula for $cap_t$ - how the capacity of a storage change in time.

The idea is simple, for a given timestamp, I need to compute the $\Delta cap_t = cap_{t+1} - cap_t$. We can decompose it into $\Delta cap_t = \Delta cap_{age, t} + \Delta cap_{cycle, t}$ (those effects are independent; if both will happen in the same time - batter will die much faster).

#### Cycle degradation
A cycle degradation is much harder topic, since each cycle decrease the capacity differently, cycles that are deeper decrease it more. Also, cycles at the beginning of the life of the storage, decrease it more, than the same cycles at the later stages.
In general, the cycle degradation depends on $dod$ (depth of discharge) of a cycle, and time in which the cycle was performed.

It is known, that the $\Delta cap_{cycle, t}$ is proportional to $(DOD_t)^{\gamma}$, where $\gamma\in[1.1, 1.3]$ (more or less). So the problem is, that it is not a linear function.

To formulate the model for a storage capacity cycle degradation, we need first to assume some level of capacity degradation at which we will decide, that the storage will stop operate. Let's call it $cap_{EOL}$ (end-of-life capacity).
Usually it is assumed to be around $80\%$ of the initial storage capacity.

Then we can assume, that the total energy throughput (in the storage lifetime) can be approximated as $T_{EOL} = margin\cdot DOD_{avg} \cdot N\cdot \frac{cap_{t_0} + cap_{EOL}}{2}$, where $N$ is the total number of cycles that storage can perform, $DOD_{avg}$ is the average
depth of storage cycle (I would assume $=0.7$, around the "sweet spot", not "maximal depth" as it causes less damage in terms of capacity degradation), $margin$ is a buffer for non-ideal conditions.

This is an approximation and assumes linear degradation (which is not true, but this is the first version, so it is acceptable at this point).

Let's define a coefficient $\alpha = \frac{\Delta cap_{cycle, total}}{T_{EOL}}$ - the ratio of total cycle capacity degradation divided by total energy throughput. This tells us, what amount of capacity (approximately) will be lost from a one unit of energy throughput.

Now, we can start by introducing some segmentation of the $[0,1]$ interval - creating capacity ranges. Let $s_i$ will be a vector of numbers from $[0,1]$, where $s_j < s_{j+1}$ and (why - it will be clear in a moment) for each $j$ we have that $P\cdot dt < cap_t\cdot (s_j - s_{j-1})$.
In other words, that ($dt$ is the time resolution, for example $0.25$ for $15$-minutes resolution) storage can not go from $soc_t = cap_t\cdot s_j$ to $soc_{t+1} > cap_t\cdot s_{j+1}$ in one timestep. We also postulate, that $s_0 = 0$ and $s_1 = 1$..

Now, let's decompose (for each $t$) $soc_t = \sum_j soc_{t,j}$ (the $j$ index is the same as for $s_j$). Next
$$
soc_{t,j} \le (s_{j+1} - s_j)\cdot cap_t
$$
$$
e_{t,j} = \sum_{j=0}^k soc_{t,j}
$$
$$
e_{j,t} \le e_{t, j+1}
$$

in that way we are sure, that if $soc_{t,j} > 0$, then all $soc_{t,l}$ are at the maximum level for all $l<j$. This will be very important property for us later.

Now, we need to define the delta state-of-charge as $\Delta soc_{t,j} \ge 0$ and $\Delta soc_{t, j} \ge soc_{t+1,j} - soc_{t,j}$.

The function $F(x) = x^{1.2}$ is non-linear, so we need to approximate the $F(DOD)\cdot soc_{max}$. We can visualize a cycle as "path upward" on the $soc(t)$ graph (where $soc(t)$ is a continuous function). When we reach the
top (the local maximum), we are on the "top" of a cycle. So the idea is this.

Let's define the following $w_j = F(s_j) - F(s_{j-1})$. The cycle degradation formula is this:

$$
\Delta cap_{cycle, t} = \alpha\cdot \sum_j w_j\cdot\Delta soc_{t,j}.
$$

Now, let's say we want to compute a degradation for a cycle, for which $soc_{max} = 0.6\cdot cap$. We can see, that no matter what path will lead us to the given $soc_{max}$, the final degradation equals $\alpha\cdot soc_{max}\cdot F(s_j)$, where $s_j$ is the end-point of the capacity
range in which $soc_{max}$ lies. This approach is much better than the industry standard used in the energy storage system models in the context of energy markets (models I find in the literature).

For most accurate approximation model, we will choose for all $j$ that $s_j - s_{j-1} = (dt + \epsilon)\cdot\frac{P}{cap}$. In that way we will enforce, that storage can not go from $soc_t = cap_t\cdot s_j$ to $soc_{t+1} > cap_t\cdot s_{j+1}$ in one timestep, and the number of
the approximation nodes will be maximal.


#### Aging degradation

Formula taken from the literature:

$$
\Delta cap_{cal, t} = f(T)_t\cdot f(SOC)_t\cdot DECAY_t,
$$

temperature component:
$$
f(T)_t = \exp\left(\frac{E_a}{R}\cdot (\frac{1}{T_{ref}} - \frac{1}{T_{amb, t} + \Delta T_{int}}) \right),
$$
where:
- $T_{ref}$ - reference temperature [K] ($298.15$ K or $~25^{\circ}C$) -- in Kelvin
- $T_{amb, t}$ - air temperature -- in Kelvin
- $\Delta T_{init}$ - constant offset representing the fact, that the internal heat BESS containers are warmer than outside air ($2-5$K) -- in Kelvin
- $R = 8.314$ (universal gas constant)
- $E_a$ - activation energy (I can assume the benchmark $=50000$) -- in $\frac{J}{mol}$

SOC component:
$$
f(SOC) = \exp(0.6\cdot SOC)
$$

time decay component:
$$
DECAY_t = \frac{B_{ref}}{2\sqrt{t}}
$$
where:
- $Q_{loss}(t) = B_{reff}\cdot \sqrt{t}$ ($t$ represent days - it is good enough approximation for my baseline 10-year model)
- it means, that $\frac{cap_{init}}{cap_{eol}}= B_{ref}\cdot\sqrt{N_{steps}}$ ($N_{steps}$ is the total number of timesteps in the storage lifetime), so $B_{ref} = \frac{cap_{init}}{cap_{eol}\cdot\sqrt{N_{steps}}}$

How to include it into the model? $f(T)_t$ can be a precomputed time series, as well as $DECAY_t$. $f(SOC)_t$ needs linear approximation - technique will be the same as the one described below for the cycle degradation modeling.


##### Improvenemnts

- try to set $\alpha = \alpha_t$, so the cycle penalization will depend on the exploitation level of the battery ($\alpha_t$ should be monotonic, and it should satisfy $\sum_t \alpha_t\cdot T_t = \sum_t \Delta cap_{cycle, t}$) - it requires time to go deep into the BESS theory
- better model of the calendar degradation; now it is extra stupid
- include air temperature to the model: degradation is different for different air temperature

probably then it will be as good as possible (it is better than most market storage models already...)

In [3]:
import os
from pathlib import Path
from src.preprocess.load import load_storage_static_params

_ROOT = Path(os.getcwd()).parent
storage_static_params_path = _ROOT / 'assets' / 'input' / 'storage_static_params.yaml'

storage_static_params = load_storage_static_params(storage_static_params_path)

#### Calendar degradation model

In [4]:
class CalendarDegradationModel:

    _R = 8.314  # universal gas constant [J/mol]
    _T_ref = 298.15  # reference temperature [K]
    _dT_init = 3.5  # constant offset (bess is warmer than the outside temp) [K] - it should depend on the storage power(t); future improvement
    _E_a = 5 * 1e4  # activation energy [J/mol]
    
    _SOC_DEG_RATE = 0.6
    
    def __init__(self, input_data: InputData) -> None:
        ssp = input_data.storage_static_params
        self.base_ref = self._compute_base_ref(ssp)
    
    def degradation(self, temperature: float, soc: float, nth_day: float) -> float:
        temperature_deg = self._temperature_degradation(temperature)
        soc_deg = self._soc_degradation(soc)
        time_deg = self._time_decay_degradation(nth_day)
        
        return  temperature_deg * soc_deg * time_deg
    
    @staticmethod
    def _compute_base_ref(sp: StorageStaticParams) -> float:
        """
        Base reference degradation rate for time decay (per day)
        """
        cal_deg_time = sp.degradation.time_decay_duration_years  # calendar decay time
        max_cap_loss = sp.degradation.max_capacity_loss  # end of life capacity

        return max_cap_loss / (np.sqrt(cal_deg_time * 365))
        
    
    def _temperature_degradation(self, temperature: float) -> float:
        """
        Temperature degradation component
        """
        return np.exp(
            (self._E_a / self._R) * (1 / self._T_ref - 1 / (temperature + self._dT_init))
        )
    
    def _soc_degradation(self, soc: float) -> float:
        """
        State of charge degradation component
        """
        return np.exp(self._SOC_DEG_RATE * soc)
    
    def _time_decay_degradation(self, nth_day: float) -> float:
        """
        Time decay component
        """
        return self.base_ref / (2 * np.sqrt(nth_day))

#### Cycle degradation model

In [ ]:
class CycleDegradationModel:

    def __init__(self, input_data: InputData) -> None:
        self.alpha = self._compute_alpha(input_data)

    def degradation(self, soc_max: float, cap_t: float) -> float:
        return self.alpha * soc_max * self._F(soc_max / cap_t)

    @staticmethod
    def _F(soc_max: float) -> float:
        """
        cycle-depth degradation function
        """
        return soc_max ** 1.2

    @staticmethod
    def _compute_alpha(input_data: InputData) -> float:
        """
        compute alpha parameter: estimated degradation per 1 unit of cycle soc_max
        """
        max_cap_loss = input_data.storage_static_params.degradation.max_capacity_loss
        cap_0 = input_data.storage_static_params.technical.capacity

        total_energy_throughput = CycleDegradationModel._compute_total_energy_throughput(input_data)

        return cap_0 * max_cap_loss / total_energy_throughput

    @staticmethod
    def _compute_total_energy_throughput(input_data: InputData) -> float:
        """
        Average (simplified) formula to estimate total energy throughput
        """
        dod_avg = input_data.storage_static_params.degradation.dod_avg
        n_cycles = input_data.storage_static_params.degradation.n_cycles
        cap_0 = input_data.storage_static_params.technical.capacity
        max_cap_loss = input_data.storage_static_params.degradation.max_capacity_loss

        cap_eol = cap_0 * (1 - max_cap_loss)

        return dod_avg * n_cycles * (cap_0 + cap_eol) / 2


#### Model Evaluation Action Plan

First:
- prepare some "random" soc_t data
- get temperature time series from entsoe (or any other source)
- draw the degradation curve and see if it looks ok


Next:
- implement the "linear approximation" model - outside opt
- see how acurate it is (compared to non-approximation model)


Finaly:
- if ok - inject the model to the opt problem
- run the simulation for 10 years and validate cap_t from opt results

In [ ]:
# draw 3 parameters of a cycle: soc_max, soc_max_duration and start_time
# soc_max ~ N(mu=0.7, sigma=0.15)
# soc_max_duration = U(0.5, 2.5) [in hours]
# start_time = U(2:00 AM, 5:00 AM)

# + assume charging and discharging with full power